# Dataset 3: Wholesale Price and MSP Data Profiling & Rescue Notebook


In [1]:
import os
import json
import re
import pandas as pd
import numpy as np


In [2]:
# Load Raw Price & MSP JSON Dataset
raw_json_path = "../data/raw/track3_price_and_msp.json"
with open(raw_json_path, 'r', encoding='utf-8') as f:
    price_json = json.load(f)
df_raw = pd.DataFrame(price_json)

In [4]:
df_raw.head(10)

,record_id,date,mandi_id,district,crop_name,min_price,max_price,modal_price,msp
0,PR001650,2026/07/26,MANDI005,None,Kapas,6850.985817820893,"₹7,570.17",7210.58,"₹6,620.00"
1,PR000246,2026-07-17,MANDI028,Fatehabad,कपास,"₹6,944.79",7654.18,"Rs. 7,299",
2,PR010091,09.01.2026,MANDI011,Jalandhar,Dhaan,"Rs. 1,857",2013.18,"₹1,935.06","INR 2,183"
3,PR000982,2026/05/24,M012,Ferozepur,corn,"₹1,873.34","Rs. 1,986","Rs. 1,930","Rs. 2,090"
4,PR001708,18/08/2026,013,Karnal,Cotton,"5,878.62/-",6574.283051769212,6226.45,"Rs. 6,620"
5,PR004312,01-24-2026,MANDI001,Patiala,Kapas,,"₹7,324.58",,"6,620.00/-"
6,PR000288,08-Aug-2026,M031,Hisar,Kanak,"₹2,270.14",2823.4429693584902,"2,546.79/-","₹2,275.00"
7,PR002536,2026-01-06,M036,Kurukshetra,WHEAT,"₹1,961.63","Rs. 2,340","INR 2,151",2275
8,PR009427,12-Aug-2026,MANDI-050,,Gehun,2277.53,"INR 2,458","₹2,367.64","2,275.00/-"
9,PR003418,09.02.2026,M011,Ambala,गन्ना,"₹3,156.94",3890.02,,"Rs. 3,500"


In [6]:
df_raw.shape

(12000, 9)

In [9]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12000 entries, 0 to 11999
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   record_id    12000 non-null  object
 1   date         12000 non-null  object
 2   mandi_id     10765 non-null  object
 3   district     11227 non-null  object
 4   crop_name    12000 non-null  object
 5   min_price    12000 non-null  object
 6   max_price    12000 non-null  object
 7   modal_price  12000 non-null  object
 8   msp          12000 non-null  object
dtypes: object(9)
memory usage: 843.9+ KB


### Step 1: Raw JSON Data Profiling and Messiness Audit

**Problem Strategy**:
1. `track3_price_and_msp.json` contains 12,000 wholesale trading records with prices and Minimum Support Prices (MSP).
2. We inspect missing location keys (`mandi_id`, `district`), messy string prices containing currency symbols (`₹`, `Rs.`, `INR`, `,`), mixed date strings (`2026/07/26`, `08-Aug-2026`), and crop name aliases.

In [10]:
# Check Raw Dimensions & Missing Values
initial_records = len(df_raw)
missing_mandi_before = df_raw['mandi_id'].isnull().sum()
missing_district_before = df_raw['district'].isnull().sum()

In [12]:
print("BEFORE Cleaning Price & MSP Dataset Scan:- ")
print("Total Raw JSON Records Loaded:", initial_records)
print("Missing mandi_id entries:", missing_mandi_before)
print("Missing district entries:", missing_district_before)

BEFORE Cleaning Price & MSP Dataset Scan:- 
Total Raw JSON Records Loaded: 12000
Missing mandi_id entries: 1235
Missing district entries: 773


In [13]:
# Check for currency symbols in the 'modal_price' column
currency_symbols_count = df_raw['modal_price'].apply(
    lambda x: bool(re.search(r'Rs|₹|INR', str(x)))
).sum()

In [14]:
commas_count = df_raw['modal_price'].apply(
    lambda x: ',' in str(x)
).sum()

In [15]:
print("Modal price records with currency symbols (₹/Rs/INR):", currency_symbols_count)
print("Modal price records with commas:", commas_count)


Modal price records with currency symbols (₹/Rs/INR): 6044
Modal price records with commas: 7176


In [16]:
print("\n--- First 10 Raw Price JSON Records ---")
df_raw.head(10)


--- First 10 Raw Price JSON Records ---


,record_id,date,mandi_id,district,crop_name,min_price,max_price,modal_price,msp
0,PR001650,2026/07/26,MANDI005,None,Kapas,6850.985817820893,"₹7,570.17",7210.58,"₹6,620.00"
1,PR000246,2026-07-17,MANDI028,Fatehabad,कपास,"₹6,944.79",7654.18,"Rs. 7,299",
2,PR010091,09.01.2026,MANDI011,Jalandhar,Dhaan,"Rs. 1,857",2013.18,"₹1,935.06","INR 2,183"
3,PR000982,2026/05/24,M012,Ferozepur,corn,"₹1,873.34","Rs. 1,986","Rs. 1,930","Rs. 2,090"
4,PR001708,18/08/2026,013,Karnal,Cotton,"5,878.62/-",6574.283051769212,6226.45,"Rs. 6,620"
5,PR004312,01-24-2026,MANDI001,Patiala,Kapas,,"₹7,324.58",,"6,620.00/-"
6,PR000288,08-Aug-2026,M031,Hisar,Kanak,"₹2,270.14",2823.4429693584902,"2,546.79/-","₹2,275.00"
7,PR002536,2026-01-06,M036,Kurukshetra,WHEAT,"₹1,961.63","Rs. 2,340","INR 2,151",2275
8,PR009427,12-Aug-2026,MANDI-050,,Gehun,2277.53,"INR 2,458","₹2,367.64","2,275.00/-"
9,PR003418,09.02.2026,M011,Ambala,गन्ना,"₹3,156.94",3890.02,,"Rs. 3,500"
